<a href="https://colab.research.google.com/github/Jofilis/chatter-tts-colab/blob/main/CCversao2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STORYAI — CHATTERBOX TTS
# Preparação do ambiente + interface
# ============================================================

import subprocess
import sys

print("========================================")
print(" STORYAI — PREPARANDO AMBIENTE")
print("========================================")

# ------------------------------------------------------------
# 1. Instala Chatterbox sem substituir o PyTorch do Colab
# ------------------------------------------------------------

print("\n[1/4] Instalando Chatterbox...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    "chatterbox-tts"
])

print("✓ Chatterbox instalado")


# ------------------------------------------------------------
# 2. Dependências auxiliares
# ------------------------------------------------------------

print("\n[2/4] Instalando dependências auxiliares...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "s3tokenizer",
    "conformer",
    "librosa",
    "soundfile",
    "safetensors",
    "huggingface_hub",
    "transformers",
    "diffusers",
    "resemble-perth"
])

print("✓ Dependências instaladas")


# ------------------------------------------------------------
# 3. Verificação da GPU
# ------------------------------------------------------------

print("\n[3/4] Verificando GPU...")

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA disponível:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA não está disponível.")

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 4. Importação do Chatterbox
# ------------------------------------------------------------

print("\n[4/4] Testando Chatterbox...")

from chatterbox.tts import ChatterboxTTS

print("✓ Chatterbox importado")

print("\n========================================")
print(" AMBIENTE PREPARADO COM SUCESSO")
print("========================================")

 STORYAI — PREPARANDO AMBIENTE

[1/4] Instalando Chatterbox...
✓ Chatterbox instalado

[2/4] Instalando dependências auxiliares...
✓ Dependências instaladas

[3/4] Verificando GPU...
PyTorch: 2.11.0+cu128
CUDA: 12.8
CUDA disponível: True
GPU: Tesla T4

[4/4] Testando Chatterbox...
✓ Chatterbox importado

 AMBIENTE PREPARADO COM SUCESSO


In [2]:
# ============================================================
# STORYAI — ETAPA 2
# Carregando modelo Chatterbox na GPU
# ============================================================

import torch
from chatterbox.tts import ChatterboxTTS

print("========================================")
print(" STORYAI — CARREGANDO MODELO")
print("========================================")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nDispositivo: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nCarregando Chatterbox...")

model = ChatterboxTTS.from_pretrained(device=device)

print("\n✓ Modelo carregado com sucesso!")
print(f"✓ Modelo está em: {device}")

print("\n========================================")
print(" MODELO PRONTO")
print("========================================")

 STORYAI — CARREGANDO MODELO

Dispositivo: cuda
GPU: Tesla T4

Carregando Chatterbox...


ve.safetensors: reconstructing file:   0%|          |  0.00B / 5.70MB            

ve.safetensors: downloading bytes:           |  0.00B            

t3_cfg.safetensors: reconstructing file:   0%|          |  0.00B / 2.13GB            

t3_cfg.safetensors: downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/25.5k [00:00<?, ?B/s]

conds.pt: reconstructing file:   0%|          |  0.00B /  107kB            

conds.pt: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:391: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000

✓ Modelo carregado com sucesso!
✓ Modelo está em: cuda

 MODELO PRONTO


In [ ]:
# ============================================================
# STORYAI — ETAPA 7
# Geração de histórias longas por partes
# ============================================================

import torch
import gradio as gr
import torchaudio
import os
import re
import shutil
import traceback
import time

# ------------------------------------------------------------
# CONFIGURAÇÕES
# ------------------------------------------------------------

OUTPUT_DIR = "/content/storyai_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("========================================")
print(" STORYAI — GERAÇÃO POR PARTES")
print("========================================")
print(f"Diretório de saída: {OUTPUT_DIR}")


# ============================================================
# 1. DIVISOR DE HISTÓRIA
# ============================================================

def dividir_historia(texto, max_chars=650):

    texto = texto.strip()

    if not texto:
        return []

    # Normaliza espaços
    texto = re.sub(r'\r\n?', '\n', texto)

    # Divide primeiro por parágrafos
    paragrafos = [
        p.strip()
        for p in texto.split("\n")
        if p.strip()
    ]

    partes = []
    atual = ""

    for paragrafo in paragrafos:

        # Se o parágrafo sozinho já for grande,
        # dividimos por frases.
        if len(paragrafo) > max_chars:

            frases = re.split(
                r'(?<=[.!?])\s+',
                paragrafo
            )

            for frase in frases:

                frase = frase.strip()

                if not frase:
                    continue

                if len(atual) + len(frase) + 1 <= max_chars:

                    if atual:
                        atual += " " + frase
                    else:
                        atual = frase

                else:

                    if atual:
                        partes.append(atual.strip())

                    atual = frase

        else:

            if len(atual) + len(paragrafo) + 1 <= max_chars:

                if atual:
                    atual += "\n" + paragrafo
                else:
                    atual = paragrafo

            else:

                if atual:
                    partes.append(atual.strip())

                atual = paragrafo

    if atual:
        partes.append(atual.strip())

    return partes


# ============================================================
# 2. JUNTAR ÁUDIOS
# ============================================================

def juntar_audios(lista_arquivos, arquivo_final):

    print("\nJuntando áudios...")

    waveforms = []
    sample_rate = None

    for arquivo in lista_arquivos:

        wav, sr = torchaudio.load(arquivo)

        if sample_rate is None:
            sample_rate = sr

        # Garante mesma taxa
        if sr != sample_rate:

            wav = torchaudio.functional.resample(
                wav,
                sr,
                sample_rate
            )

        waveforms.append(wav)

    if not waveforms:
        raise RuntimeError("Nenhum áudio para juntar.")

    audio_final = torch.cat(
        waveforms,
        dim=1
    )

    torchaudio.save(
        arquivo_final,
        audio_final,
        sample_rate
    )

    print(f"✓ Áudio completo salvo em:")
    print(arquivo_final)

    return arquivo_final


# ============================================================
# 3. GERADOR PRINCIPAL
# ============================================================

def gerar_historia(
    audio_referencia,
    historia,
    cfg_weight,
    exaggeration,
    temperature,
    progresso=gr.Progress()
):

    try:

        # ----------------------------------------------------
        # VALIDAÇÕES
        # ----------------------------------------------------

        if audio_referencia is None:

            return (
                None,
                None,
                "❌ Envie um áudio de referência."
            )

        if not historia or not historia.strip():

            return (
                None,
                None,
                "❌ Digite uma história."
            )

        # ----------------------------------------------------
        # LIMPAR OUTPUT ANTERIOR
        # ----------------------------------------------------

        if os.path.exists(OUTPUT_DIR):

            for arquivo in os.listdir(OUTPUT_DIR):

                caminho = os.path.join(
                    OUTPUT_DIR,
                    arquivo
                )

                if os.path.isfile(caminho):

                    os.remove(caminho)

        # ----------------------------------------------------
        # DIVIDIR HISTÓRIA
        # ----------------------------------------------------

        partes = dividir_historia(
            historia,
            max_chars=650
        )

        total = len(partes)

        if total == 0:

            return (
                None,
                None,
                "❌ Não foi possível dividir a história."
            )

        print("\n========================================")
        print(" HISTÓRIA DIVIDIDA")
        print("========================================")

        print(f"Caracteres: {len(historia)}")
        print(f"Partes: {total}")

        for i, parte in enumerate(partes, 1):

            print(
                f"Parte {i}: "
                f"{len(parte)} caracteres"
            )

        # ----------------------------------------------------
        # GERAÇÃO
        # ----------------------------------------------------

        arquivos_gerados = []

        for i, parte in enumerate(partes, 1):

            progresso(
                (i - 1) / total,
                desc=f"Gerando parte {i}/{total}"
            )

            print("\n----------------------------------------")
            print(f"GERANDO PARTE {i}/{total}")
            print("----------------------------------------")

            print(parte)

            inicio = time.time()

            # ------------------------------------------------
            # CHATTERBOX
            # ------------------------------------------------

            wav = model.generate(
              parte,
              audio_prompt_path=audio_referencia,

              repetition_penalty=1.2,
              min_p=0.05,
              top_p=1.0,

              exaggeration=float(exaggeration),
              cfg_weight=float(cfg_weight),
              temperature=float(temperature)
            )

            # ------------------------------------------------
            # SALVAR
            # ------------------------------------------------

            arquivo_parte = os.path.join(
                OUTPUT_DIR,
                f"parte_{i:03d}.wav"
            )

            torchaudio.save(
                arquivo_parte,
                wav.cpu(),
                model.sr
            )

            arquivos_gerados.append(
                arquivo_parte
            )

            tempo = time.time() - inicio

            print(
                f"✓ Parte {i} salva "
                f"({tempo:.1f}s)"
            )

            # Atualiza progresso
            progresso(
                i / total,
                desc=f"Parte {i}/{total} concluída"
            )

        # ----------------------------------------------------
        # JUNTAR
        # ----------------------------------------------------

        progresso(
            0.99,
            desc="Juntando áudios..."
        )

        arquivo_final = os.path.join(
            OUTPUT_DIR,
            "historia_completa.wav"
        )

        juntar_audios(
            arquivos_gerados,
            arquivo_final
        )

        # ----------------------------------------------------
        # LISTA DE ARQUIVOS
        # ----------------------------------------------------

        lista = "\n".join(
            [
                f"✓ {os.path.basename(x)}"
                for x in arquivos_gerados
            ]
        )

        status = (
            f"✅ HISTÓRIA CONCLUÍDA!\n\n"
            f"Partes geradas: {total}\n"
            f"Caracteres: {len(historia)}\n\n"
            f"Arquivos:\n{lista}\n\n"
            f"✓ historia_completa.wav"
        )

        progresso(
            1.0,
            desc="Concluído!"
        )

        return (
            arquivo_final,
            arquivo_final,
            status
        )

    except Exception as e:

        print("\n========================================")
        print("❌ ERRO DURANTE A GERAÇÃO")
        print("========================================")

        traceback.print_exc()

        return (
            None,
            None,
            f"❌ Erro durante a geração:\n\n{str(e)}"
        )


# ============================================================
# 4. INTERFACE GRADIO
# ============================================================

with gr.Blocks(
    title="StoryAI TTS"
) as app_storyai:

    gr.Markdown(
        """
        # 🎙️ StoryAI TTS

        ### Geração de histórias narradas
        """
    )

    gr.Markdown(
        """
        Envie uma amostra da voz e cole sua história.
        O sistema dividirá o texto automaticamente e
        salvará cada parte antes de gerar a próxima.
        """
    )

    # --------------------------------------------------------
    # VOZ
    # --------------------------------------------------------

    audio_referencia = gr.Audio(
        label="🎙️ Voz de referência",
        type="filepath"
    )

    # --------------------------------------------------------
    # HISTÓRIA
    # --------------------------------------------------------

    historia = gr.Textbox(
        label="📖 História",
        placeholder="Cole sua história em inglês aqui...",
        lines=20
    )

        # --------------------------------------------------------
    # CONFIGURAÇÕES DE VOZ
    # --------------------------------------------------------

    gr.Markdown("### ⚙️ Configurações de voz")

    cfg_weight = gr.Slider(
        minimum=0.2,
        maximum=0.8,
        value=0.5,
        step=0.05,
        label="CFG Weight"
    )

    exaggeration = gr.Slider(
        minimum=0.0,
        maximum=1.0,
        value=0.5,
        step=0.05,
        label="Exaggeration"
    )

    temperature = gr.Slider(
        minimum=0.1,
        maximum=1.5,
        value=0.8,
        step=0.05,
        label="Temperature"
    )

    # --------------------------------------------------------
    # BOTÃO
    # --------------------------------------------------------

    botao_gerar = gr.Button(
        "🎙️ GERAR HISTÓRIA",
        variant="primary"
    )

    # --------------------------------------------------------
    # PROGRESSO
    # --------------------------------------------------------

    gr.Markdown(
        """
        ### Progresso
        """
    )

    # --------------------------------------------------------
    # STATUS
    # --------------------------------------------------------

    status = gr.Textbox(
        label="Status",
        lines=12,
        interactive=False
    )

    # --------------------------------------------------------
    # RESULTADO
    # --------------------------------------------------------

    audio_final = gr.Audio(
        label="🔊 História completa",
        type="filepath"
    )

    arquivo_download = gr.File(
        label="⬇️ Baixar história completa"
    )

    # --------------------------------------------------------
    # EVENTO
    # --------------------------------------------------------

    botao_gerar.click(
    fn=gerar_historia,
    inputs=[
        audio_referencia,
        historia,
        cfg_weight,
        exaggeration,
        temperature
    ],
    outputs=[
        audio_final,
        arquivo_download,
        status
    ]
)


# ============================================================
# 5. INICIAR
# ============================================================

print("\n========================================")
print(" STORYAI PRONTO")
print("========================================")

app_storyai.launch(
    share=True,
    debug=True
)

 STORYAI — GERAÇÃO POR PARTES
Diretório de saída: /content/storyai_output

 STORYAI PRONTO
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d11d9fc112e281e5ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
